# Compare Parameter Sweeps

Cross-instrument, cross-timeframe, and cross-strategy comparison of
saved sweep results from `data/sweeps/`.  No backtests are re-run — all
inputs are the v2 Parquet files dropped by `run_sweep()`.

**Prerequisites:** Run a sweep in any `notebooks/backtest/*.ipynb`
notebook at least once.  Each `run_sweep()` call writes to
`data/sweeps/{strategy}_{instrument}_{interval}.parquet`.


## 1. Setup


### 1.1 Imports & shared config

All editable knobs live here.  Cell 1.2 onward derives everything from
this one cell — only touch them when you know what you're changing.


In [ ]:
# ── Imports ────────────────────────────────────────────────────
import pandas as pd
from IPython.display import display

# Add notebooks/ to sys.path so charts.py + utils.py imports resolve.
import sys
sys.path.insert(0, str(__import__("pathlib").Path(".").resolve()))

from charts import generate_cross_sweep_html, plot_pnl_heatmap
from utils import load_sweeps_filtered, save_notebook_snapshot

# Notebook-private helpers (extracted to keep cells short).
from _compare_helpers import build_stability_df, short_sweep_label


# ─────────────────────────────────────────────────────────────────
# Filters — narrow the cross-sweep view (None = no filter)
# ─────────────────────────────────────────────────────────────────
STRATEGY      : str | None = None      # e.g. "MACross-EMA"
INSTRUMENT_ID : str | None = None      # e.g. "BTC-USD-PERP.HYPERLIQUID"
BAR_INTERVAL  : str | None = None      # e.g. "1h"

# ─────────────────────────────────────────────────────────────────
# Row filters — drop noise that pollutes ranking + heatmaps
# ─────────────────────────────────────────────────────────────────
FILTER_LIQUIDATED = True   # drop liquidated rows
FILTER_SPOTLIGHT  = True   # drop _kind == "spotlight" rows

# ─────────────────────────────────────────────────────────────────
# Param columns to use for stability + heatmaps
# ─────────────────────────────────────────────────────────────────
# These must be the strategy's grid params (e.g. fast/slow for MA
# strategies).  Stability + heatmap cells silently skip sweeps that
# don't have all of these columns.
PARAM_COLS = ["fast", "slow"]
ROW_COL    = "slow"   # heatmap y-axis
COL_COL    = "fast"   # heatmap x-axis

# ─────────────────────────────────────────────────────────────────
# Ranking
# ─────────────────────────────────────────────────────────────────
# Column to rank "best params per sweep" by.  Use total_pnl_pct for
# fair cross-instrument comparison (pnl-pct normalises away starting
# capital and notional differences).  Other good choices: mar_ratio,
# recovery_factor.
RANK_BY = "total_pnl_pct"

# ─────────────────────────────────────────────────────────────────
# File naming
# ─────────────────────────────────────────────────────────────────
RESULT_NAME = "compare_sweeps"
SAVE_ON_RUN_ALL = True

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("Imports OK")


### 1.2 Load sweeps + filter

Reads every matching parquet via `load_sweeps_filtered`, which:

- Warns on schema-version mismatch (current target: v2).
- Drops liquidated rows (uses the v2 `liquidated` bool, falls back to
  the v1 `error == "liquidated"` check on older sweeps).
- Drops `_kind == "spotlight"` rows so off-grid combos don't pollute
  rankings or heatmaps.

Toggle either filter via `FILTER_LIQUIDATED` / `FILTER_SPOTLIGHT` in
1.1 if you want to inspect them.


In [ ]:
sweeps = load_sweeps_filtered(
    strategy=STRATEGY,
    instrument_id=INSTRUMENT_ID,
    bar_interval=BAR_INTERVAL,
    filter_liquidated=FILTER_LIQUIDATED,
    filter_spotlight=FILTER_SPOTLIGHT,
)

if not sweeps:
    print("No sweeps loaded.  Run a backtest notebook with run_sweep() first.")
else:
    # load_sweeps_filtered already printed the count + filter summary.
    # Add a tighter per-sweep view with combo counts + data ranges.
    print()
    print("Per-sweep coverage:")
    for label, df in sweeps.items():
        if df.empty:
            print(f"  {label:55s}  (empty after filters)")
            continue
        data_start = df.get("_data_start", pd.Series([""])).iloc[0]
        data_end   = df.get("_data_end",   pd.Series([""])).iloc[0]
        rng = f"{str(data_start)[:10]} → {str(data_end)[:10]}"
        print(f"  {label:55s}  ({len(df):4d} combos, {rng})")


## 2. Best params per sweep


### 2.1 Best params table

For each sweep, picks the row with the highest `RANK_BY` (default
`total_pnl_pct`) and shows the v2 metric columns.  Sharpe / Sortino
are intentionally absent — see `docs/ANALYZER_RETURNS_CAVEAT.md`.


In [ ]:
if not sweeps:
    print("No sweeps loaded.")
else:
    rows = []
    for label, df in sweeps.items():
        valid = df[df["total_pnl"].notna()]
        if valid.empty:
            continue
        best = valid.loc[valid[RANK_BY].idxmax()]

        row: dict = {"sweep": label}
        if "_data_start" in best.index and "_data_end" in best.index:
            row["data_range"] = (
                f"{str(best['_data_start'])[:10]} → {str(best['_data_end'])[:10]}"
            )
        for pc in PARAM_COLS:
            if pc in best.index:
                row[f"best_{pc}"] = best[pc]

        # v2 metric columns (skip any not present in this sweep)
        for col in [
            "total_pnl", "total_pnl_pct", "num_positions",
            "win_rate", "pnl_profit_factor", "expectancy",
            "max_drawdown_pct", "mar_ratio", "recovery_factor", "cagr",
        ]:
            if col in best.index:
                row[col] = best[col]

        rows.append(row)

    if rows:
        summary = pd.DataFrame(rows).sort_values(RANK_BY, ascending=False)
        print(f"=== Best params per sweep (ranked by {RANK_BY}) ===")
        display(summary)
    else:
        print("No valid results in any loaded sweep.")


### 2.2 Sortable HTML cross-sweep table

Renders one combined sortable table containing every row from every
loaded sweep.  Sort by any v2 metric (PnL%, MAR, recovery factor, …)
and filter by the **Sweep** column to drill into one strategy/
instrument/timeframe.

Output goes to `reports/sweeps/<RESULT_NAME>.html` — open it in a
browser for full interactivity (DataTables.js).


In [ ]:
if not sweeps:
    print("No sweeps loaded.")
else:
    cross_html_path = generate_cross_sweep_html(
        sweeps,
        filename=RESULT_NAME,
    )
    print(f"\nOpen in browser: {cross_html_path}")


## 3. Side-by-side PnL heatmaps

One heatmap per sweep using `total_pnl_pct` so heatmaps are comparable
across instruments with different starting capital and notional.
Spotlight rows have already been filtered upstream; we still pass
`exclude_kinds=("spotlight",)` defensively in case the user disables
the global filter.


In [ ]:
if not sweeps:
    print("No sweeps loaded.")
else:
    for label, df in sweeps.items():
        if ROW_COL not in df.columns or COL_COL not in df.columns:
            print(f"Skipping {label} — missing '{ROW_COL}' or '{COL_COL}'")
            continue
        if df.duplicated(subset=[ROW_COL, COL_COL]).any():
            print(
                f"Skipping {label} — duplicate ({COL_COL}, {ROW_COL}) pairs "
                "(extra params in sweep)"
            )
            continue
        plot_pnl_heatmap(
            df,
            row_col=ROW_COL, col_col=COL_COL,
            value_col="total_pnl_pct",
            row_label=f"{ROW_COL.title()} Period",
            col_label=f"{COL_COL.title()} Period",
            title=f"PnL% — {short_sweep_label(label)}",
            fmt=".1f",
            exclude_kinds=("spotlight",),
        )


## 4. Parameter stability across sweeps


### 4.1 Stability table

For each `(fast, slow)` combo, averages PnL% across every sweep that
contains it.  Combos profitable across **all** sweeps are robust;
combos that only work on one timeframe / instrument are the canonical
overfitting tell.


In [ ]:
if not sweeps:
    print("No sweeps loaded.")
elif len(sweeps) < 2:
    print("Need 2+ sweeps to analyse parameter stability.")
    print("Run your backtest notebook for another instrument or timeframe,")
    print("then re-run this notebook.")
else:
    stability, n_sweeps = build_stability_df(sweeps, PARAM_COLS)
    if stability.empty:
        print("No sweeps contain all PARAM_COLS — nothing to analyse.")
    else:
        robust = stability[
            (stability["sweep_count"] == n_sweeps)
            & stability["all_profitable"]
        ]
        if robust.empty:
            print(f"No combos profitable across all {n_sweeps} sweeps.")
            print()
            print(f"Top 10 by average PnL% (across all sweeps):")
            display(stability.head(10))
        else:
            print(f"✅ {len(robust)} combos profitable across all {n_sweeps} sweeps:")
            display(robust.head(20))


### 4.2 Average PnL% heatmap

Heatmap of per-combo average PnL% across every sweep that has full
coverage.  Bright stable regions (e.g. a contiguous green block) are
the robust ones; isolated single-cell winners are likely overfit.


In [ ]:
if not sweeps or len(sweeps) < 2 or len(PARAM_COLS) != 2:
    print("Stability heatmap requires 2+ sweeps and exactly 2 PARAM_COLS.")
else:
    stability, n_sweeps = build_stability_df(sweeps, PARAM_COLS)
    if stability.empty:
        print("No stability data to plot.")
    else:
        full_coverage = stability[stability["sweep_count"] == n_sweeps]
        if full_coverage.empty:
            print("No combos appear in all sweeps — nothing to plot.")
        else:
            plot_pnl_heatmap(
                full_coverage,
                row_col=ROW_COL, col_col=COL_COL,
                value_col="avg_pnl_pct",
                row_label=f"{ROW_COL.title()} Period",
                col_label=f"{COL_COL.title()} Period",
                title=f"Avg PnL% across {n_sweeps} sweeps",
                fmt=".1f",
            )


## 5. Single-sweep deep dive

Pick a sweep by index and explore its top combos with the v2 metric
columns.  Useful as a launchpad into a per-strategy notebook for the
combos worth following up.


In [ ]:
if not sweeps:
    print("No sweeps loaded.")
else:
    print("Available sweeps:")
    for i, label in enumerate(sweeps.keys()):
        print(f"  [{i}] {label}")

    # ── Change this to pick a different sweep ──
    PICK = 0

    pick_label = list(sweeps.keys())[PICK]
    pick_df = sweeps[pick_label]
    print()
    print(f"Inspecting: {pick_label}")
    print(f"Top 10 by {RANK_BY}:")

    # Only include columns that exist in this sweep.
    param_cols_present = [c for c in PARAM_COLS if c in pick_df.columns]
    metric_cols = [c for c in [
        "total_pnl", "total_pnl_pct", "num_positions",
        "win_rate", "pnl_profit_factor", "max_drawdown_pct",
        "mar_ratio", "recovery_factor", "cagr",
    ] if c in pick_df.columns]
    display(
        pick_df
        .sort_values(RANK_BY, ascending=False)
        .head(10)
        [param_cols_present + metric_cols]
    )

    # ── Next step pointer ─────────────────────────────────────────
    # Low trade counts (< ~20) make per-sweep "best" combos
    # statistically weak — high MAR / recovery on a thin sample is
    # often overfit.  validate_strategy.ipynb runs plateau detection,
    # walk-forward, and bootstrap CI on the chosen combo to surface
    # those failure modes.
    print()
    print("→ Next step: open notebooks/validate_strategy.ipynb,")
    print("  set STRATEGY / ASSET / BAR_INTERVAL to match this sweep,")
    print("  and run the full validation stack before paper trading.")


## 6. Save & cleanup


### 6.1 Save snapshot

Saves a self-contained `.ipynb` + rendered `.html` snapshot to
`reports/notebooks/compare/` and `reports/html/compare/` respectively.
Honours the `SAVE_ON_RUN_ALL` flag from cell 1.1.


In [ ]:
# Trailing semicolon suppresses Jupyter's auto-display of the
# returned Path (the helper already prints "Saved -> ..." for both
# .ipynb and .html).
save_notebook_snapshot(
    "compare_sweeps.ipynb", RESULT_NAME,
    save_on_run_all=SAVE_ON_RUN_ALL,
    category="compare",
);


### 6.2 Cleanup

Nothing to dispose — this notebook only reads parquets, no engine.


In [ ]:
print("Done.")


## 7. Scratchpad

Ad-hoc analysis goes here.  Anything below this line is excluded from
the canonical layout.


In [ ]:
# Scratchpad — your ad-hoc analysis goes here.
